In [0]:
%pip install xgboost scikit-learn

In [0]:
%restart_python

In [0]:
import json
import pickle
import mlflow
import mlflow.pyfunc
import pandas as pd

In [0]:
class FraudModelWrapper(mlflow.pyfunc.PythonModel):
    def __init__(self, threshold):
        self.threshold = threshold

    def load_context(self, context):
        with open(context.artifacts["model"], "rb") as f:
            self.model = pickle.load(f)
        with open(context.artifacts["encoders"], "rb") as f:
            self.encoders = pickle.load(f)

    def predict(self, context, model_input: pd.DataFrame) -> pd.Series:
        df = model_input.copy()
        for col, le in self.encoders.items():
            df[col] = le.transform(df[col].astype(str))
        proba = self.model.predict_proba(df)[:, 1]
        return pd.Series((proba >= self.threshold).astype(int))

In [0]:
with open("/Volumes/credit_transactions/bronze/reference_data/threshold.json") as f:
    threshold_value = json.load(f)["threshold"]

input_example = pd.read_csv("/Volumes/credit_transactions/bronze/reference_data/input_example.csv")

mlflow.pyfunc.log_model(
    artifact_path="fraud_model",
    python_model=FraudModelWrapper(threshold=threshold_value),
    artifacts={
        "model": "/Volumes/credit_transactions/bronze/reference_data/fraud_model.pkl",
        "encoders": "/Volumes/credit_transactions/bronze/reference_data/encoders.pkl",  
    },
    registered_model_name="credit_transactions.gold.fraud_model",
    input_example=input_example
)